# Unit 5 Exercise: The Strategy Pattern in Password Storage

Ian Stevens, Advanced Object-Oriented Design and Programming, Unit 5.

The unit asks for the Strategy pattern to be implemented in a project, so that an algorithm can be chosen and replaced without editing the code that uses it. I chose password storage. The recommended hashing algorithm and its cost settings change over time, and a real system has to keep verifying old hashes while it writes new ones with the current choice, so the ability to swap an algorithm is a requirement rather than a nicety.

The code is in `password_hashing.py` and the tests are in `test_password_hashing.py`. Everything uses the standard library only.

In [1]:
import ast
import hashlib
import textwrap
import inspect
import platform
import time

import password_hashing
from password_hashing import (HashFormatError, HashingStrategy, LegacySha256, PasswordStore,
                              Pbkdf2Sha256, Scrypt, main)

print("Python", platform.python_version())

Python 3.13.18


## 1. The three roles

Nesteruk (2022, ch. 22) describes Strategy as defining the skeleton of an algorithm and using composition to supply the missing details. Here the skeleton is `PasswordStore.check_password()`: find the stored hash, verify it, and replace it if it is out of date. The details, how a password becomes a hash, belong to the strategies. `HashingStrategy` is the abstract strategy, `Pbkdf2Sha256`, `Scrypt` and `LegacySha256` are the concrete strategies, and `PasswordStore` is the context.

In [2]:
for cls in (HashingStrategy, Pbkdf2Sha256, Scrypt, LegacySha256):
    ident = getattr(cls, "identifier", "(abstract)")
    print(f"{cls.__name__:<16} parent={cls.__mro__[1].__name__:<16} identifier={ident}")

try:
    HashingStrategy()
except TypeError as error:
    print("\nTypeError:", error)

HashingStrategy  parent=ABC              identifier=(abstract)
Pbkdf2Sha256     parent=HashingStrategy  identifier=pbkdf2_sha256
Scrypt           parent=HashingStrategy  identifier=scrypt
LegacySha256     parent=HashingStrategy  identifier=sha256

TypeError: Can't instantiate abstract class HashingStrategy without an implementation for abstract methods 'hash', 'verify'


Each stored hash names the strategy and settings that produced it, so one store can hold hashes from several strategies at the same time.

In [3]:
for strategy in (Pbkdf2Sha256(iterations=1_000), Scrypt(n=2**10)):
    encoded = strategy.hash("correct horse battery staple")
    print(f"{strategy!r:<34} {encoded[:60]}...")

Pbkdf2Sha256(iterations=1000)      pbkdf2_sha256$1000$VOpjBy5nHTfm5B/RnET18Q==$Z7Bu4izCe7gjZCK2...
Scrypt(n=2**10, r=8, p=1)          scrypt$1024$8$1$V1/oKYekqwkDL764WyoJsg==$XZr6mseU0on/01oZ+M9...


## 2. Swapping the strategy while the program runs

`main()` follows one store through a migration. Alice's hash is imported from an older system that used unsalted SHA-256. When she logs in, her password is verified with `LegacySha256` and immediately rehashed with the current strategy. The strategy is then switched to scrypt without restarting anything; Bob registers with scrypt, and Alice moves to scrypt at her next login. The demonstration lowers the cost settings so that it runs instantly.

In [4]:
main();

alice imported from the old system     alice=sha256
alice logs in                          alice=pbkdf2_sha256
strategy switched; bob registers       alice=pbkdf2_sha256, bob=scrypt
alice logs in again                    alice=scrypt, bob=scrypt


## 3. What the pattern replaces

Without the pattern, the obvious design is a function that branches on the algorithm name. It works, but every new algorithm means editing it, and the branch has to be repeated in every function that depends on the algorithm (hashing, verifying and deciding whether to rehash). The context in this design has no branch on an algorithm name and names no concrete strategy, which the tests check by parsing its source.

In [5]:
def hash_password(algorithm, password, salt):
    if algorithm == "pbkdf2_sha256":
        return hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 600_000)
    elif algorithm == "scrypt":
        return hashlib.scrypt(password.encode(), salt=salt, n=2**17, r=8, p=1, maxmem=2**28)
    elif algorithm == "sha256":
        return hashlib.sha256(password.encode()).digest()
    raise ValueError("unknown algorithm")

def branches(obj):
    tree = ast.parse(textwrap.dedent(inspect.getsource(obj)))
    return sum(isinstance(n, ast.If) for n in ast.walk(tree))

store_source = inspect.getsource(PasswordStore)
print("if statements in hash_password():", branches(hash_password))
print("concrete strategies named in PasswordStore:",
      [n for n in ("Pbkdf2Sha256", "Scrypt", "LegacySha256") if n in store_source])

if statements in hash_password(): 3
concrete strategies named in PasswordStore: []


The choice has not disappeared, though. `PasswordStore._strategy_for()` still reads the identifier from the stored hash and looks it up. Nesteruk's own dynamic strategy has the same feature: his `set_output_format()` uses a `switch` to choose a strategy (2022, ch. 22). Strategy removes the branching from the algorithm, and leaves one lookup in one place.

In [6]:
print(inspect.getsource(PasswordStore._strategy_for))

    def _strategy_for(self, encoded: str) -> HashingStrategy:
        identifier = encoded.split("$", 1)[0]
        strategy = self._accepted.get(identifier)
        if strategy is None:
            raise HashFormatError(f"no accepted strategy for {identifier!r}")
        return strategy



## 4. Stored hashes are untrusted input

Because the store picks a strategy from data, the data decides which code runs and how much work it does. A database row that has been tampered with could name an algorithm the store never meant to accept, or ask for a trillion PBKDF2 iterations or gigabytes of scrypt memory. The store accepts only strategies it was given, and each strategy checks the settings it reads before doing any work.

In [7]:
pbkdf2 = Pbkdf2Sha256(iterations=1_000)
store = PasswordStore(pbkdf2)
good = pbkdf2.hash("pw").split("$")
attempts = {
    "unknown algorithm": "md5$5f4dcc3b5aa765d61d8327deb882cf99",
    "unsalted SHA-256 not accepted": "sha256$" + hashlib.sha256(b"pw").hexdigest(),
    "trillion iterations": "$".join([good[0], "1000000000000", good[2], good[3]]),
}
for label, encoded in attempts.items():
    start = time.perf_counter()
    try:
        store.import_hash("mallory", encoded)
        pbkdf2.verify("pw", encoded)
    except HashFormatError as error:
        print(f"{label:<31} refused in {1000 * (time.perf_counter() - start):.2f} ms: {error}")

unknown algorithm               refused in 0.01 ms: no accepted strategy for 'md5'
unsalted SHA-256 not accepted   refused in 0.00 ms: no accepted strategy for 'sha256'
trillion iterations             refused in 0.01 ms: iterations 1000000000000 is outside 1000..10000000


An unknown user name is handled by verifying the password against a dummy hash, so that a login for a user who does not exist costs the same hash as one for a user who does. A store that returned at once would let an attacker discover which user names exist by timing the response.

## 5. The real cost settings

The defaults follow the OWASP Password Storage Cheat Sheet: PBKDF2-HMAC-SHA256 with 600,000 iterations, and scrypt with N = 2^17, r = 8 and p = 1 (OWASP, no date). The timings below are for this machine. A slow hash is the point: each guess an attacker makes against a stolen hash costs the same.

In [8]:
for strategy in (Pbkdf2Sha256(), Scrypt()):
    start = time.perf_counter()
    encoded = strategy.hash("correct horse battery staple")
    hashed = time.perf_counter() - start
    print(f"{strategy!r:<32} hash took {hashed * 1000:6.0f} ms; verifies: "
          f"{strategy.verify('correct horse battery staple', encoded)}")

Pbkdf2Sha256(iterations=600000)  hash took     98 ms; verifies: True


Scrypt(n=2**17, r=8, p=1)        hash took    400 ms; verifies: True


## 6. Strategy or Template Method?

Nesteruk (2022, ch. 23) says the two patterns do the same thing, defining the skeleton of an algorithm in one place and the details in another, and differ only in that Strategy uses composition and Template Method uses inheritance. He concludes that Template Method 'can only be static'. The difference shows here: with inheritance, a store would be a `Pbkdf2PasswordStore` or a `ScryptPasswordStore` for its whole life, and switching algorithm would mean building a new store and copying the records across. With composition, the same store object changes strategy in one call.

In [9]:
store = PasswordStore(Pbkdf2Sha256(iterations=1_000))
before = id(store)
store.set_strategy(Scrypt(n=2**10))
print("same store object:", id(store) == before, "| current strategy:", store.strategy)

same store object: True | current strategy: Scrypt(n=2**10, r=8, p=1)


## 7. Where the pattern stops helping

`LegacySha256` is a `HashingStrategy` that refuses to produce a hash, because unsalted SHA-256 should only ever be read so that it can be replaced. That makes it a strategy that cannot stand in for the others, which is a breach of the Liskov substitution principle from Unit 2. Nesteruk (2022, ch. 14) meets the same problem with commands that cannot be undone and suggests splitting the interface. Here the split would be a `Verifier` that every strategy implements and a `Hasher` that only current strategies implement.

In [10]:
store = PasswordStore(LegacySha256())
try:
    store.set_password("alice", "pw")
except NotImplementedError as error:
    print("NotImplementedError:", error)

NotImplementedError: unsalted SHA-256 is accepted for migration only


Nesteruk (2022, ch. 22) also shows strategies passed as plain functions, as in `sort()` with a comparison lambda. In Python that would be the natural choice for a single operation, but a hashing strategy has three operations and an identifier that must stay together, so here a class earns its place.

## 8. Running the tests

In [11]:
import io
import unittest
import test_password_hashing

def flatten(suite):
    for item in suite:
        yield from flatten(item) if isinstance(item, unittest.TestSuite) else [item]

suite = unittest.defaultTestLoader.loadTestsFromModule(test_password_hashing)
names = [test.id().replace("test_password_hashing.", "") for test in flatten(suite)]
result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
print(f"{result.testsRun} tests run, {len(result.failures)} failures, {len(result.errors)} errors")
groups = {}
for name in names:
    groups.setdefault(name.split(".")[0], 0)
    groups[name.split(".")[0]] += 1
for group, count in groups.items():
    print(f"  {group:<24} {count}")

32 tests run, 0 failures, 0 errors
  BriefRequirements        5
  Migration                5
  OpenForExtension         2
  RecordedLimits           3
  StoredDataIsUntrusted    9
  StrategyContract         7
  password_hashing         1


## References

Nesteruk, D. (2022) *Design patterns in modern C++20: reusable approaches for object-oriented software design*. 2nd edn. Berkeley, CA: Apress.

OWASP (no date) *Password storage cheat sheet*. OWASP Cheat Sheet Series. Available at: https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html (Accessed: 27 November 2026).